# Cardiac Patient Monitoring System
## 08 — Findings & Limitations (Phase 8)

This notebook synthesizes results already produced and saved by notebooks 01–07 (no new modeling
here) into the project's final findings and limitations narrative. Every number below is loaded
directly from the saved output files — none are re-typed from memory, per Project Quality Rule 1.


In [1]:
import json
import pandas as pd

with open("../outputs/results/phase4_baseline_metrics.json") as f:
    baseline_metrics = json.load(f)

phase5_comparison = pd.read_csv("../outputs/results/phase5_model_comparison.csv", index_col=0)

with open("../outputs/results/phase5_cv_results.json") as f:
    cv_results = json.load(f)

with open("../outputs/results/phase6_pipeline_metrics.json") as f:
    pipeline_results = json.load(f)

print("Loaded all saved results.")


Loaded all saved results.


In [2]:
phase5_comparison


,accuracy,precision,recall,f1,roc_auc,cv_f1_mean,cv_f1_std
Logistic Regression,0.8689,0.8125,0.9286,0.8667,0.9578,0.8254,0.0150
Random Forest,0.9016,0.8438,0.9643,0.9000,0.9545,0.7917,0.0374


In [3]:
pd.DataFrame({
    'Logistic Regression (+FE)': pipeline_results['Logistic Regression (+FE)'],
    'Random Forest (+FE)': pipeline_results['Random Forest (+FE)'],
}).T.round(4)


,accuracy,precision,recall,f1,roc_auc
Logistic Regression (+FE),0.8689,0.8125,0.9286,0.8667,0.9578
Random Forest (+FE),0.8852,0.8387,0.9286,0.8814,0.9556


## 1. Key Findings

### 1.1 Dataset findings
- 303 patients, 13 predictor features, from the UCI Cleveland Heart Disease database.
- Original target (`num`) has 5 classes; binarized to `target` (0 = absence, 1 = presence) per the
  documented transformation — 164 absence (54.1%) vs 139 presence (45.9%), a near-balanced split.
- Only 6 missing values total, confined to `ca` (4) and `thal` (2) — a very clean dataset overall.
- No duplicate rows, no invalid/out-of-range coded values found.

### 1.2 EDA findings
- Strongest numerical associations with the target: `thalach` (r ≈ −0.42, lower max heart rate →
  higher disease likelihood) and `oldpeak` (r ≈ +0.42, larger ST depression → higher disease
  likelihood).
- Strongest categorical signals: `cp` (chest pain type, especially category 4/asymptomatic),
  `thal` (especially category 7/reversible defect), and `exang` (exercise-induced angina).
- `chol`, `trestbps`, and `oldpeak` are right-skewed with legitimate high-end outliers (kept, not
  removed).
- No severe multicollinearity among numerical features.
- Demographic imbalance in `sex` (~68% male), inherited from the source data.

### 1.3 Supervised-learning findings


In [4]:
summary_table = phase5_comparison[['accuracy','precision','recall','f1','roc_auc','cv_f1_mean','cv_f1_std']]
summary_table


,accuracy,precision,recall,f1,roc_auc,cv_f1_mean,cv_f1_std
Logistic Regression,0.8689,0.8125,0.9286,0.8667,0.9578,0.8254,0.0150
Random Forest,0.9016,0.8438,0.9643,0.9000,0.9545,0.7917,0.0374


- **Baseline (Logistic Regression):** accuracy 0.869, precision 0.813, recall 0.929, F1 0.867,
  ROC-AUC 0.958. Strong baseline — the underlying signal is largely linearly separable.
- **Comparison model (Random Forest):** accuracy 0.902, precision 0.844, recall 0.964, F1 0.900,
  ROC-AUC 0.955 — better on every single-split test metric, but with **higher cross-validation
  variance** (CV F1 mean 0.792, std 0.037) than Logistic Regression (CV F1 mean 0.825, std 0.015).
  This is a genuine trade-off: Random Forest looks stronger on this particular test split, but
  Logistic Regression is more consistently stable across different training folds.
- **After feature engineering (Phase 6):** the two engineered features (`hr_reserve_ratio`,
  `bp_category`) did **not** improve results — Random Forest's test F1 actually dropped slightly
  (0.900 → 0.881) and Logistic Regression was essentially unchanged (F1 0.867 → 0.867). This null
  result is reported as-is rather than omitted.
- **Confusion-matrix observation:** for both models, false negatives (missed disease cases) were
  kept low relative to false positives — consistent with the project's explicit prioritization of
  recall, since a missed diagnosis is more costly than an unnecessary follow-up test.

### 1.4 Unsupervised findings
- K-Means with `k=2` (chosen via elbow + silhouette) produced two clusters: an older, lower-`thalach`,
  higher-`oldpeak` cluster (n=131) with 77.1% target-presence proportion, and a younger, higher-
  `thalach`, lower-`oldpeak` cluster (n=172) with 22.1% target-presence proportion.
- The silhouette score at k=2 was modest (~0.175) — weak-to-moderate cluster separation, not tight
  geometric clusters. Reported honestly rather than overstated.
- PCA to 2 components captured only ~38.9% of total variance, yet the 2D projection still showed a
  visible left/right split that lined up with both the K-Means clusters and, more loosely, the
  known target — a useful cross-check that the unsupervised structure is not disconnected from the
  supervised signal, without claiming the clustering diagnoses anything.


## 2. Model Conclusion

Based on the evidence above — not accuracy alone — **Random Forest is the stronger candidate on
this test split** (higher accuracy, precision, recall, and F1), and its recall in particular
(0.929–0.964 across runs) is the most clinically relevant metric given the project's explicit
framing that false negatives are the costlier error. However, **Logistic Regression showed more
stable cross-validation performance**, meaning its single-split advantage for Random Forest should
not be overstated as a guaranteed generalization advantage — with only 303 rows and a 61-row test
set, single-split differences of a few percentage points carry real uncertainty.

**Practical recommendation:** Random Forest is selected as the final pipeline artifact
(`models/cardiac_pipeline.pkl`) primarily for its consistently higher recall across both the
Phase 5 and Phase 6 (post-feature-engineering) runs, while explicitly flagging its higher CV
variance as a caveat rather than hiding it.


## 3. Limitations

- **Sample size:** 303 patients is small for machine learning; a 61-row test set means individual
  misclassifications shift metrics by ~1.6 percentage points each — reported metrics should be
  read as estimates with real uncertainty, not precise figures.
- **Single-source data:** only the Cleveland database is used (not the Hungary/Switzerland/VA Long
  Beach subsets in the broader UCI repository), and only from one hospital system circa 1988 —
  findings may not generalize to other populations, eras, or measurement protocols.
- **Demographic imbalance:** ~68% male in the source data; model performance for female patients is
  based on proportionally fewer examples and has not been separately validated.
- **Feature engineering yielded no measurable improvement** in this run — the two engineered
  features did not outperform the raw features, which itself is a useful (if modest) finding about
  this particular dataset/model combination, not a project shortfall.
- **Modest unsupervised structure:** the K-Means silhouette score (~0.175) indicates the natural
  cluster structure in this feature space is weak-to-moderate, not sharply separated.
- **Model limitations:** Logistic Regression assumes roughly linear/additive effects; Random Forest,
  while more flexible, showed higher variance across CV folds on this small dataset and is more
  prone to overfitting without the `max_depth=5` constraint already applied here.
- **Evaluation limitations:** metrics come from a single stratified train/test split plus 5-fold
  CV; a fully robust estimate would use repeated CV or nested CV, which was out of scope for this
  curriculum-aligned project.
- **Non-clinical nature:** this project is an educational ML analysis. It does not provide
  diagnosis, treatment recommendations, or emergency guidance, and none of its outputs (models,
  clusters, metrics) should be interpreted as clinical decision support.


## 4. Save consolidated findings document


In [5]:
findings_md = '''# Cardiac Patient Monitoring System — Findings & Limitations

## Dataset
- 303 patients, 13 features, UCI Cleveland Heart Disease database.
- Target binarized: 164 absence (54.1%) / 139 presence (45.9%).
- 6 missing values total (ca: 4, thal: 2); no duplicates; no invalid values.

## Supervised Learning Results
{comparison_table}

## After Feature Engineering
{pipeline_table}

## Model Conclusion
Random Forest selected as the final pipeline artifact for consistently higher recall
(0.929-0.964), the priority metric given this project's medical framing (false negatives are
costlier than false positives), while Logistic Regression showed more stable cross-validation
performance (lower std) as an explicit caveat.

## Unsupervised Learning
K-Means (k=2, chosen via elbow + silhouette) produced clusters with 77.1% vs 22.1% disease-presence
proportions -- a meaningful post-hoc alignment with the known target, though the clusters were
formed without using the target. Silhouette score was modest (~0.175).

## Key Limitations
- Small sample size (303 rows, 61-row test set) -- metrics carry real uncertainty.
- Single-source, single-era data (Cleveland only, ~1988) -- limited generalizability.
- ~68% male demographic imbalance in source data.
- Feature engineering did not improve results in this run.
- Educational project only -- not a clinical diagnostic tool.
'''.format(
    comparison_table=phase5_comparison.to_markdown(),
    pipeline_table=pd.DataFrame({
        'Logistic Regression (+FE)': pipeline_results['Logistic Regression (+FE)'],
        'Random Forest (+FE)': pipeline_results['Random Forest (+FE)'],
    }).T.round(4).to_markdown()
)

with open("../outputs/results/phase8_findings_and_limitations.md", "w") as f:
    f.write(findings_md)

print(findings_md)


# Cardiac Patient Monitoring System — Findings & Limitations

## Dataset
- 303 patients, 13 features, UCI Cleveland Heart Disease database.
- Target binarized: 164 absence (54.1%) / 139 presence (45.9%).
- 6 missing values total (ca: 4, thal: 2); no duplicates; no invalid values.

## Supervised Learning Results
|                     |   accuracy |   precision |   recall |     f1 |   roc_auc |   cv_f1_mean |   cv_f1_std |
|:--------------------|-----------:|------------:|---------:|-------:|----------:|-------------:|------------:|
| Logistic Regression |     0.8689 |      0.8125 |   0.9286 | 0.8667 |    0.9578 |       0.8254 |      0.015  |
| Random Forest       |     0.9016 |      0.8438 |   0.9643 | 0.9    |    0.9545 |       0.7917 |      0.0374 |

## After Feature Engineering
|                           |   accuracy |   precision |   recall |     f1 |   roc_auc |
|:--------------------------|-----------:|------------:|---------:|-------:|----------:|
| Logistic Regression (+FE) |  

## Phase 8 Quality Gate — Checklist

- [x] Dataset findings summarized
- [x] EDA findings summarized
- [x] Supervised-learning findings summarized, loaded from saved results (no invented numbers)
- [x] Unsupervised findings summarized
- [x] Model conclusion stated, evidence-based, trade-offs disclosed honestly
- [x] Limitations documented (sample size, single-source data, demographic imbalance, feature
      engineering null result, model/evaluation limitations, non-clinical nature)
- [x] Consolidated findings document saved to
      `outputs/results/phase8_findings_and_limitations.md`

**Next:** Phase 9 — Documentation + Reproducibility (README, requirements, final notebook
cleanup) — Milestone M7.
